# RSIAT on Kaggle

Notebook này chạy RSIAT trên Kaggle GPU. Mã nguồn được clone vào `/kaggle/working`, dữ liệu đầu vào đọc từ `/kaggle/input` (nếu đã attach Kaggle Dataset), còn log và checkpoint được ghi vào `/kaggle/working/RSIAT_outputs`. Bật Internet trong Kaggle nếu cần tải CIFAR-100 hoặc trọng số ViT pretrained.

In [ ]:
# Cập nhật danh sách gói và cài đặt Python 3.10
!apt-get update -y
!apt-get install python3.10 python3.10-distutils -y

# Tải và cài đặt pip riêng cho Python 3.10
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py

In [ ]:
!python3.10 -m pip install uv

In [ ]:
!python3.10 -m uv pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!python3.10 -m uv pip install alembic==1.16.4 certifi==2025.8.3 charset-normalizer==3.4.3 colorlog==6.9.0 contourpy==1.3.2 cycler==0.12.1 easydict==1.13
!python3.10 -m uv pip install filelock==3.13.1 fonttools==4.59.2 fsspec==2024.6.1 greenlet==3.2.4 hf-xet==1.1.8 huggingface-hub==0.34.4 idna==3.10 Jinja2==3.1.4 joblib==1.5.2 kiwisolver==1.4.9 llvmlite==0.44.0 Mako==1.3.10
!python3.10 -m uv pip install MarkupSafe==2.1.5 matplotlib==3.10.6 mpmath==1.3.0 networkx==3.3 numba==0.61.2 numpy==2.1.2 nvidia-cublas-cu12==12.6.4.1
!python3.10 -m uv pip install nvidia-cuda-cupti-cu12==12.6.80 nvidia-cuda-nvrtc-cu12==12.6.77 nvidia-cuda-runtime-cu12==12.6.77 nvidia-cudnn-cu12==9.10.2.21 nvidia-cufft-cu12==11.3.0.4 nvidia-cufile-cu12==1.11.1.6
!python3.10 -m uv pip install nvidia-curand-cu12==10.3.7.77 nvidia-cusolver-cu12==11.7.1.2 nvidia-cusparse-cu12==12.5.4.2 nvidia-cusparselt-cu12==0.7.1 nvidia-nccl-cu12==2.27.3 nvidia-nvjitlink-cu12==12.6.85 nvidia-nvtx-cu12==12.6.77
!python3.10 -m uv pip install optuna==4.5.0 packaging==25.0 pandas==2.3.2 pillow==11.0.0 pynndescent==0.5.13 pyparsing==3.2.3 python-dateutil==2.9.0.post0 pytz==2025.2 PyYAML==6.0.2 requests==2.32.5 scikit-learn==1.7.2
!python3.10 -m uv pip install timm==0.6.12 kaggle

In [ ]:
%cd /kaggle/working/RSIAT

In [ ]:
from pathlib import Path
import os
import subprocess
import urllib.error
import urllib.request

from torchvision.datasets import CIFAR100

REPO_URL = 'https://github.com/deety03503/QG_test.git'
BRANCH = 'main'
PROJECT_DIR = Path('/kaggle/working/RSIAT')
OUTPUT_ROOT = Path('/kaggle/working/RSIAT_outputs')
DATA_ROOT = Path('/kaggle/working/RSIAT_data/datasets')
CIFAR_DIR = DATA_ROOT / 'cifar-100-python'
CIFAR_FILES = ('train', 'test', 'meta')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.environ['RSIAT_DATA_ROOT'] = str(DATA_ROOT)
os.chdir(PROJECT_DIR)


def has_cifar100():
    return CIFAR_DIR.is_dir() and all((CIFAR_DIR / filename).is_file() for filename in CIFAR_FILES)


if not has_cifar100():
    print('Đang tải CIFAR-100 vào', DATA_ROOT)
    try:
        urllib.request.urlopen('https://www.cs.toronto.edu', timeout=15).close()
    except (OSError, urllib.error.URLError) as error:
        raise RuntimeError(
            'Không thể tải CIFAR-100. Hãy bật Internet trong Kaggle Settings.'
        ) from error
    CIFAR100(root=str(DATA_ROOT), train=True, download=True)
    CIFAR100(root=str(DATA_ROOT), train=False, download=True)

if not has_cifar100():
    raise RuntimeError(f'CIFAR-100 chưa hoàn tất, kiểm tra thư mục: {CIFAR_DIR}')

print('CIFAR-100 đã sẵn sàng:', CIFAR_DIR)
print('Project:', PROJECT_DIR)
print('Data:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
import torch, torchvision
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('torchvision:', torchvision.__version__)

# Kaggle images usually include torch/torchvision; install only project dependencies.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt
import timm
print('timm:', timm.__version__)
!python -m py_compile main.py trainer.py models/base.py models/RSIAT_adapter.py data/data.py

In [ ]:
import json

from torchvision.datasets import CIFAR100

assert CIFAR_DIR.is_dir(), f'Không tìm thấy CIFAR-100: {CIFAR_DIR}'
cifar_train = CIFAR100(root=str(DATA_ROOT), train=True, download=False)
cifar_test = CIFAR100(root=str(DATA_ROOT), train=False, download=False)
print('CIFAR-100:', len(cifar_train), 'train /', len(cifar_test), 'test')


def make_kaggle_config(source, destination, seed=1993, resume=False, max_tasks_per_run=None):
    config = json.loads((PROJECT_DIR / source).read_text())
    config.update({
        'seed': [seed],
        'resume': resume,
        'output_root': str(OUTPUT_ROOT),
        'device': ['0'] if torch.cuda.is_available() else [-1],
        'num_workers': 2,
        'stats_num_workers': 2,
        'persistent_workers': True,
    })
    if max_tasks_per_run is not None:
        config['max_tasks_per_run'] = max_tasks_per_run
    Path(destination).write_text(json.dumps(config, indent=2))

make_kaggle_config('exps/adapter_cifar224_smoke.json', '/kaggle/working/RSIAT/rsiat_smoke.json', max_tasks_per_run=1)
make_kaggle_config('exps/adapter_cifar224.json', '/kaggle/working/RSIAT/rsiat_full.json', resume=True)

## Smoke test

Smoke test chạy task đầu tiên với 1 epoch, kiểm tra data, pretrained ViT, train, evaluation và checkpoint trên Kaggle.

In [ ]:
!python3.10 main.py --config /kaggle/working/RSIAT/rsiat_smoke.json

## Full CIFAR-100 experiment

Config full bật `resume`: checkpoint và log nằm trong `/kaggle/working/RSIAT_outputs`. Nếu muốn giữ kết quả sau khi phiên Kaggle kết thúc, hãy bật Save Version hoặc xuất thư mục output thành Kaggle Dataset.

In [ ]:
# Bỏ comment sau khi smoke test thành công.
!python3.10 main.py --config /kaggle/working/RSIAT/rsiat_full.json

In [ ]:
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs' / 'adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)